## 기본구조 3

##### Output parser

지금까지는 LLM에게 프롬르트로 질문한 것이다. 그럼 답변은? 답변의 형식도 지정해야 한다. 

내가 준 지시사항에 단순히 너의 페르소나 외에도 아웃풋 포맷을 지정해줄 수 있다. 예를 들어 답변은 반드시 리스트 또는 json으로 해줘! 라고 프롬프트로 강제로 넣어줘서 질무을 하면, LLM이 답변 형태도 잘 정해준다는 것이다. 다만, 내가 사용할 모델이 강력한 LLM 모델이 아니라면(경량 모델) 그런 모델로 동일한 프롬프트를 적어서 질문을 하면? -> 뜬금없어지는 경우가 많다. 이런 경우 나의 로직과 맞지 않는다

이를 방지하기 위해서 랭체인 안에서는 ouput parser를 따로 지정할 수 있다. 

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field # Field는 메타데이터 붙이는 도구 # Pydantic = 파이썬 데이터가 내가 정한 구조와 타입에 맞는지 검사해주는 라이브러리
from typing import List, Optional
import json


In [ ]:
# 1. 출력 스키마 정의 (이게 핵심!)  # 내가 받고 싶은 답변들의 특징과 형태까지 지정한다.
class MeetingSummary(BaseModel):
    title: str = Field(description="회의 제목") # Field는 Pydantic 모델에서 각 필드의 추가 정보를 설정하는 함수. 즉, 이 필드에 대한 설명은 덧붙인다.
    date: str = Field(description="회의 날짜 (YYYY-MM-DD)")
    attendees: List[str] = Field(description="참석자 목록")
    key_decisions: List[str] = Field(description="주요 결정사항 3~5개")
    action_items: List[str] = Field(description="후속 조치 사항")
    next_meeting: Optional[str] = Field(default=None, description="다음 회의 일정")

# 2. 파서 생성
parser = PydanticOutputParser(pydantic_object=MeetingSummary)

# 3. 프롬프트에 format_instructions 자동 삽입
prompt = ChatPromptTemplate.from_messages([
    ("system", """
너는 전문 비서다. 회의 내용을 다음 JSON 형식으로 요약해줘.
반드시 이 형식만 출력하고, 다른 설명은 절대 하지 마.

{format_instructions}
"""),
    ("human", "{meeting_notes}")
])

# format_instructions:LLM에게 “답변을 어떤 형식으로 출력해야 하는지” 알려주는 지시문


In [ ]:
# LLM
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# 체인 구성
chain = prompt | llm | parser # 이번에는 파서까지 체이닝 하는 것이다. 그럼 일련의 과정은? 프롬프트 LLM에게 넣어주고 파서를 토해서 LLM이 답변을 하라! 이렇게 체이닝이 되는 것이다. 

# 실행
result = chain.invoke({ # invoke를 통해 LLM에게 질문을 함. 
    "meeting_notes": """
    2025년 3월 10일 마케팅팀 회의
    참석: 김팀장, 이과장, 박대리, 최사원
    주요 논의: Q2 캠페인 예산 30% 증액 결정
    인플루언서 5명 추가 계약
    다음 회의: 3월 24일 오후 2시
    """,
    "format_instructions": parser.get_format_instructions()
})

print(result) # 내가 원하는 스키마로 답변이 구조화되어서 나온다. 이렇게 LLM의 출력 형태를 구조화할 수 있다. ouput parser를 통해 출력 스키마 정의하기!
# 이렇게 구조화된 출력을 가져야 그 이후의 로직을 이거에 맞춰서 쓸 수 있을 것이다.


# "format_instructions": parser.get_format_instructions() 코드에서는 파서가 원하는 출력 형식 안내문을 만들어서 프롬프트에 넣어준다. 
# 위의 코드를  실행하면 LLM이 어떤 구조로 답해야 하는지 설명하는 지시문이 자동으로 생성된다. 
# 마지막 체이닝에서 파서가 파싱할 수 있도록 이런 구조로 답해라! 라고 지시하는 역할을 한다. 즉, 파싱하고 싶은 구조에 맞는 출력 지시문을 생성함

 1. Pydantic 모델로 원하는 결과 구조를 정의한다
  2. parser.get_format_instructions()가 그 구조에 맞는 출력 지시문을 만든다
  3. 그 지시문을 프롬프트에 넣어 LLM에게 전달한다
  4. LLM이 JSON처럼 파싱 가능한 형식으로 답변한다
  5. parser가 그 답변을 실제 Pydantic 객체로 변환한다

  즉, format_instructions 자체가 파싱을 하는 건 아니다. 

## chain = prompt | llm | parser 진행 순서 보기

  1. prompt 단계

  chain.invoke({...})에 넣은 값들이 프롬프트 템플릿에 들어갑니다.

```python
{
"meeting_notes": "...회의 내용...",
"format_instructions": parser.get_format_instructions()
}
```

  여기서:

  - {meeting_notes}에는 회의 원문이 들어감
  - {format_instructions}에는 MeetingSummary 구조에 맞춰 JSON으로 답하라는 지시문이 들어감

  즉, 최종적으로 LLM에게 보낼 메시지가 만들어집니다.

  ———

  2. llm 단계

  완성된 프롬프트가 Gemini 모델로 전달됩니다.

```python
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
```

  Gemini는 회의 내용을 읽고, format_instructions에 맞춰 JSON 형태의 답변을 생성합니다.

  예를 들면 내부적으로 이런 식의 응답을 만들려고 합니다.

```python
{
"title": "마케팅팀 회의",
"date": "2025-03-10",
"attendees": ["김팀장", "이과장", "박대리", "최사원"],
"key_decisions": [
"Q2 캠페인 예산 30% 증액",
"인플루언서 5명 추가 계약"
```
    ],
```python
"action_items": [],
"next_meeting": "2025-03-24 14:00"
}
```

  ———

  3. parser 단계

  LLM이 만든 JSON 문자열을 PydanticOutputParser가 받아서 MeetingSummary 객체로 변환합니다.

  이때 Pydantic이 타입을 검사합니다.

```python
title: str
date: str
attendees: List[str]
key_decisions: List[str]
action_items: List[str]
next_meeting: Optional[str]
```

  형식이 맞으면 최종 결과는 그냥 문자열이 아니라, MeetingSummary 타입의 구조화된 객체가 됩니다.

  그래서 이런 식으로 접근할 수 있습니다.

```python
print(result.title)
print(result.date)
print(result.attendees)
print(result.next_meeting)
```

  ———

  전체 흐름을 한 줄로 요약하면:

  입력값 → 프롬프트 완성 → Gemini가 JSON 답변 생성 → parser가 Pydantic 객체로 변환

  그래서:

```python
chain = prompt | llm | parser
```

  는

  프롬프트 만들고 → LLM에게 답변 받고 → 원하는 구조로 파싱한다

  는 의미입니다.


In [ ]:
# CommaSeparatedOutputParser는 언어 모델의 출력을 쉼표로 구분된 값으로 파싱하는 데 사용됩니다.
# 이 클래스는 언어 모델이 생성한 텍스트 응답을 쉼표로 구분된 형식으로 변환하여, 이를 쉽게 처리하고 분석할 수 있도록 합니다.

from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# 콤마로 구분된 리스트 출력 파서 초기화
output_parser = CommaSeparatedListOutputParser() # 이 방식은 LangChain에 이미 만들어져 있는 기본 파서를 가져와 쓰는 방식이다. 

# 출력 형식 지침 가져오기
format_instructions = output_parser.get_format_instructions()

# 프롬프트 템플릿 설정
prompt = PromptTemplate(
    template="List five {subject}.\n{format_instructions}",
    input_variables=["subject"],
    partial_variables={"format_instructions": format_instructions},
)
# subject는 나중에 호출할 때 직접 넣는 값
# format_instructions는 프롬프트 생성 시 미리 고정해두는 값
# partial_variables는 반복해서 넣기 번거로운 값으로 항상 같은 값으로 들어가는 값이다.  “템플릿의 일부 변수 값을 미리 채워두는 기능

# 프롬프트, 모델, 출력 파서를 연결하여 체인 생성
chain = prompt | llm | output_parser

# "대한민국 관광명소"에 대한 체인 호출 실행
chain.invoke({"subject": "대한민국 서울 관광명소"})


In [ ]:
llm.invoke('대한민국 서울 관광명소') # 그냥 LLM에게 답변하기. 차이가 있다!

In [ ]:
chain2 = prompt | llm  # 프롬프트 체이닝해서 질문하기 
chain2.invoke('대한민국 서울 관광명소') 

# 프롬프트에 따라서 5개의 답변은 했다. 그런데, outputparse 내용이랑 비교해보면? LLM이 하나의 문장으로 해주는게 아니라 정확하게 내가 알고싶은 주제만 ,로 구분되는 리스트 형태로 준거랑 다르다 .내가 원한 답변이 리스트로 구분된 답변을 원하면CommaSeparatedListOutputParser를 쓰는게 좋겠지

# 출력 예시: AIMessage(content='경복궁, N서울타워, 명동, 북촌한옥마을, 동대문디자인플라자', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019de2a3-8579-7cb1-a568-b208fcc7a521-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 38, 'output_tokens': 194, 'total_tokens': 232, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 167}})

다른 예시

In [ ]:
# StructuredOutputParser는 언어 모델의 출력을 구조화된 데이터로 변환하는 데 사용됩니다.
# 이 클래스는 언어 모델이 생성한 텍스트 응답을 사전 정의된 데이터 모델에 맞게 변환하여, 이를 쉽게 처리하고 활용할 수 있도록 합니다.
# dict 형식으로 구성하고 key/value 쌍으로 데이터를 반환합니다. 로컬 모델과 같은 덜 강력한 모델에서도 유용함 
# 로컬 모델에서는 Pydantic 파서가 동작하지 않는 상황이 빈번하게 발생할 수 있음

from langchain_classic.output_parsers import ResponseSchema, StructuredOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# 사용자의 질문에 대한 답변
response_schemas = [
    ResponseSchema(name="answer", description="사용자의 질문에 대한 답변"),
    ResponseSchema(
        name="source",
        description="사용자의 질문에 답하기 위해 사용된 `출처`, `웹사이트주소` 이여야 합니다.",
    ),
]
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)

format_instructions = output_parser.get_format_instructions()
prompt = PromptTemplate(
    template="answer the users question as best as possible.\n{format_instructions}\n{question}",
    input_variables=["question"],
    partial_variables={"format_instructions": format_instructions},
)

chain = prompt | llm | output_parser
chain.invoke({"question": "대한민국의 수도는 어디인가요?"})

# 출력예시: {'answer': '대한민국의 수도는 서울입니다.',
 'source': 'https://ko.wikipedia.org/wiki/%EB%8C%80%ED%95%9C%EB%AF%BC%EA%B5%AD'}

**결국 파서를 지정하는 목적은 LLM의 출력 형식을 내가 원하는 구조로 통일하는 것이다** 

아웃풋을 파싱하지 않으면 LLM은 매번 답변 형식이 달라질 수 있다. 그래서 우리는 구조를 미리 정해두고 LLM의 답변을 일정한 형식으로 받거나 또는 특정 데이터 형식으로 받거나 할 수 있다.


핵심:  파서 = LLM의 자유로운 답변을 내가 정한 데이터 구조로 맞춰주는 장치입니다.

*추가*

```python
parser = PydanticOutputParser(pydantic_object=MeetingSummary)
```

  이 방식은 내가 원하는 출력 구조를 직접 정의하는 방식입니다.

```python
class MeetingSummary(BaseModel):  
title: str  
date: str  
attendees: List[str]  
...
```

  처럼 필드 이름, 타입, 설명을 직접 정합니다.
  그래서 결과를 이런 구조로 받을 수 있습니다.

  result.title
  result.date
  result.attendees

  복잡한 구조화 출력에 적합합니다.

  예:

  - 회의록 요약
  - 이력서 정보 추출
  - 상품 정보 추출
  - 뉴스 기사 분석
  - 고객 문의 분류 + 이유 + 긴급도 추출

  ———

  반면:

```python
output_parser = CommaSeparatedListOutputParser()  (또는 StructuredOutputParser)
```

  이건 LangChain에 이미 만들어져 있는 기본 파서를 가져와 쓰는 방식입니다.

  이 파서는 이름 그대로, LLM 출력이 쉼표로 구분된 리스트라고 가정하고 파싱합니다.

  예를 들어 LLM이 이렇게 답하면:

  사과, 바나나, 포도

  파서가 이렇게 바꿔줍니다.

```python
["사과", "바나나", "포도"]
```

  즉, 간단한 리스트 출력에 적합합니다.

  예:

```python
from langchain_core.output_parsers import CommaSeparatedListOutputParser

output_parser = CommaSeparatedListOutputParser()

format_instructions = output_parser.get_format_instructions()

prompt = PromptTemplate.from_template("""
```
  과일 3개를 추천해줘.

```python
{format_instructions}
""")

chain = prompt | llm | output_parser

result = chain.invoke({
"format_instructions": format_instructions
})

print(result)
```

  결과 예:

```python
["사과", "바나나", "포도"]
```

  정리하면:

  - PydanticOutputParser: 내가 직접 스키마를 정의하는 커스텀 구조화 파서 (더 엄격하게 관리할 때 사용함)
  - CommaSeparatedListOutputParser: 쉼표로 구분된 리스트를 처리하는 기본 제공 파서
  - 복잡한 구조는 PydanticOutputParser
  - 단순 리스트는 CommaSeparatedListOutputParser
  - 간단한 json 구조 빠르게 만들 때 사용 StructuredOutputParser

  그래서 네 말대로, PydanticOutputParser는 내가 하나하나 설정하는 방식, CommaSeparatedListOutputParser()는 이미 만들어진 파서를 불러와 쓰는 방식이라고 보면 됩니다.